In [0]:
-- First we need to ensure your analytical business layer schema namespace is initialized
CREATE DATABASE IF NOT EXISTS hive_metastore.f1_presentation;


In [0]:
CREATE OR REPLACE TABLE hive_metastore.f1_presentation.race_results_master
AS
SELECT 
    r.race_year,
    r.name AS race_name,
    r.race_date,
    c.name AS circuit_name,
    c.location AS circuit_city,
    c.country AS circuit_country,
    d.first_name AS driver_first_name,
    d.last_name AS driver_last_name,
    d.driver_number,
    d.nationality AS driver_nationality,
    t.team_name,
    t.nationality AS team_nationality,
    current_timestamp() AS presentation_ingestion_date
FROM hive_metastore.f1_transformed.races r
LEFT JOIN hive_metastore.f1_transformed.circuits c ON r.circuit_id = c.circuit_id
-- For training datasets, these display a structural cross-join base metric mapping
CROSS JOIN hive_metastore.f1_transformed.drivers d 
CROSS JOIN hive_metastore.f1_transformed.constructors t;

-- View the newly aggregated master layer
SELECT * FROM hive_metastore.f1_presentation.race_results_master LIMIT 50;


Which Country Has Produced the Most F1 Drivers?

In [0]:
CREATE OR REPLACE VIEW hive_metastore.f1_presentation.v_driver_nationality_distribution
AS
SELECT 
    driver_nationality,
    COUNT(DISTINCT concat(driver_first_name, ' ', driver_last_name)) AS total_unique_drivers
FROM hive_metastore.f1_presentation.race_results_master
GROUP BY driver_nationality
ORDER BY total_unique_drivers DESC;

-- Display analytical report view
SELECT * FROM hive_metastore.f1_presentation.v_driver_nationality_distribution;
